# 🐾 V16 — Animal Sound Generator (1000+ samples/class)

**More data + latent diffusion.**

| Step | What | Time |
|------|------|------|
| 1 | Collect data (1000+/class) | ~20 min |
| 2 | Train Decoder | ~30 min |
| 3 | Train Latent Diffusion | ~30 min |
| 4 | Generate & Download | ~2 min |

Data sources: ESC-50 + UrbanSound8K + Xeno-Canto

In [ ]:
# @title 1. Setup
!git clone https://github.com/weseegod/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator
!git pull

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib tqdm soundfile pydub
!apt-get install -q ffmpeg

!mkdir -p models data/esc50

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# @title 2. Collect Training Data (target: 1000+ per class)
import os, sys, json, time, shutil, csv, tarfile, urllib.request, urllib.parse
from pydub import AudioSegment

DATA = "data/animal1000"
os.makedirs(DATA, exist_ok=True)

# ── 2a. Get existing ESC-50 ──
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /tmp/esc50.zip
!unzip -qo /tmp/esc50.zip -d /tmp/
!python src/scripts/setup_esc50.py --source /tmp/ESC-50-master/audio --target data/esc50

for cls in ['Dog','Cat','Rooster','Frog','Crow','Insect','Hen']:
    os.makedirs(f"{DATA}/{cls}", exist_ok=True)
    src_dir = f"data/esc50/{cls}"
    if os.path.isdir(src_dir):
        for f in os.listdir(src_dir):
            if f.endswith('.wav'):
                shutil.copy2(f"{src_dir}/{f}", f"{DATA}/{cls}/{f}")

print("ESC-50 copied")
for cls in ['Dog','Cat','Rooster','Frog','Crow','Insect','Hen']:
    n = len([f for f in os.listdir(f'{DATA}/{cls}') if f.endswith(('.wav','.mp3'))])
    print(f"  {cls}: {n}")

# ── 2b. UrbanSound8K → Dog barks (~1,000) ──
print("\n🐕 UrbanSound8K (Dog)...")
if not os.path.exists("/tmp/UrbanSound8K"):
    !wget -q https://zenodo.org/records/1203745/files/UrbanSound8K.tar.gz -O /tmp/us8k.tar.gz
    with tarfile.open("/tmp/us8k.tar.gz") as tar:
        tar.extractall(path="/tmp/")

us8k_meta = "/tmp/UrbanSound8K/metadata/UrbanSound8K.csv"
if os.path.exists(us8k_meta):
    with open(us8k_meta) as f:
        for row in csv.DictReader(f):
            if 'dog' in row['class'].lower():
                src = f"/tmp/UrbanSound8K/audio/fold{row['fold']}/{row['slice_file_name']}"
                dst = f"{DATA}/Dog/_{row['slice_file_name']}"
                if os.path.exists(src) and not os.path.exists(dst):
                    shutil.copy2(src, dst)
    print(f"  Dog: {len(os.listdir(f'{DATA}/Dog'))} files")

# ── 2c. Xeno-Canto → Crow, Rooster, Hen, Frog ──
print("\n🐦 Xeno-Canto (birds + frog)...")
XC_QUERIES = {
    'Crow': 'Corvus corone',
    'Rooster': 'Gallus gallus',
    'Hen': 'Gallus gallus',
    'Frog': 'frog',
}

for cls, query in XC_QUERIES.items():
    print(f"  {cls}: '{query}'...")
    url = f"https://xeno-canto.org/api/2/recordings?query={urllib.parse.quote(query)}"
    try:
        with urllib.request.urlopen(url) as r:
            data = json.loads(r.read())
        recs = data.get('recordings', [])
        print(f"    Found {len(recs)} recordings")
        
        target = max(0, 500 - len([f for f in os.listdir(f'{DATA}/{cls}') if f.endswith(('.wav','.mp3'))]))
        downloaded = 0
        for rec in recs:
            if downloaded >= target:
                break
            audio_url = rec.get('file','')
            if not audio_url.startswith('http'):
                continue
            fname = f"xc_{rec['id']}.mp3"
            outpath = f"{DATA}/{cls}/{fname}"
            if os.path.exists(outpath):
                downloaded += 1
                continue
            try:
                urllib.request.urlretrieve(audio_url, outpath)
                downloaded += 1
                if downloaded % 100 == 0:
                    print(f"      {downloaded}/{target}")
                time.sleep(0.3)
            except Exception as e:
                pass
        
        # Convert mp3 to wav
        for f in os.listdir(f"{DATA}/{cls}"):
            if f.endswith('.mp3'):
                try:
                    audio = AudioSegment.from_mp3(f"{DATA}/{cls}/{f}")
                    audio = audio.set_frame_rate(22050).set_channels(1)
                    audio.export(f"{DATA}/{cls}/{f.replace('.mp3','.wav')}", format="wav")
                    os.remove(f"{DATA}/{cls}/{f}")
                except:
                    os.remove(f"{DATA}/{cls}/{f}")
                    
        n = len([f for f in os.listdir(f'{DATA}/{cls}') if f.endswith('.wav')])
        print(f"    ✅ {cls}: {n} files")
    except Exception as e:
        print(f"    ❌ {e}")

# ── Summary ──
print(f"\n{'='*40}")
print("📊 Data Summary")
for cls in ['Dog','Cat','Rooster','Frog','Crow','Insect','Hen']:
    n = len([f for f in os.listdir(f'{DATA}/{cls}') if f.endswith('.wav')])
    bar = '█'*(n//20)
    print(f"  {cls:<12}: {n:4d} {bar}")
print(f"\n📁 {DATA}/")
print("🎯 Target: 1000/class")

In [ ]:
# @title 3. Phase 1: Train Decoder
!python src/latent_diff/train_decoder.py

In [ ]:
# @title 4. Phase 2: Train Latent Diffusion
!python src/latent_diff/train_diff.py

In [ ]:
# @title 5. Generate & Download
!python src/latent_diff/generate.py

import zipfile, os
with zipfile.ZipFile('v16_animals.zip', 'w') as z:
    for f in os.listdir('outputs/generated'):
        if f.endswith('.wav'): z.write(f'outputs/generated/{f}', f)
from google.colab import files
files.download('v16_animals.zip')